In [ ]:
import numpy as np
from tqdm import tqdm

# Import all components
%run ../layers/layer.ipynb
%run ../activation/Activation.ipynb
%run ../Function/functions.ipynb
%run ../optimizer/optimizer.ipynb

#**CONNECTING LAYERS**
* Abstracting every layer above inside the Neural Network layer so that it acts like an orchectrator and helps creating the forward pass an backpropogation

In [ ]:


class NeuralNetwork:
  # ... (init and other methods) ...
  def __init__(self, layer_dims, activation_type):
    self.activation_type = activation_type
    self.modules = []
    for i in range(len(layer_dims) - 1):
      linear_layer = Linear(layer_dims[i], layer_dims[i+1])
      self.modules.append(linear_layer)
      if i < len(layer_dims) - 2 and activation_type in ['relu', 'tanh', 'sigmoid']:
        self.modules.append(Activation(activation_type))

  def _feed_forward(self, input_data):
    out = input_data
    for module in self.modules:
      out = module(out)
    return out

  def _backward(self, grad):
    for module in reversed(self.modules):
      grad = module.backward(grad)
    return grad

  def parameters(self):
    params = []
    for module in self.modules:
      if isinstance(module, Linear):
        params.append(module)
    return params

  def compile(self,optimizer,loss, metrics):
    pass

  def fit(self,x, y, epochs, learning_rate, batch_size, dataset, num_classes):
    # Optimizer should be created ONCE, before the epoch loop
    optimizer_instance = SGD(self.parameters(), learning_rate) # Instantiated once

    for _ in range(epochs):
      print(f"\nEpoch {_+1}/{epochs} :")
      total_loss = 0
      num_batches = 0
      for batch_start in tqdm(range(0, len(x),batch_size)):
        batch_end = batch_start + batch_size
        batch_x = x.iloc[batch_start:batch_end].values
        batch_y = y.iloc[batch_start:batch_end].values

        logits = self._feed_forward(batch_x)

        fn=Function()
        prob=fn.softmax(logits)

        one_hot_y = np.zeros((len(batch_y), num_classes))
        one_hot_y[np.arange(len(batch_y)), batch_y.flatten()] = 1.0

        batch_y_reshaped_for_loss = batch_y.reshape(-1, 1)
        current_loss = crossentropy(prob, batch_y_reshaped_for_loss)
        total_loss += current_loss
        num_batches += 1

        grad = (np.array(prob) - one_hot_y) / batch_size
        self._backward(grad)

        # Use the optimizer_instance created once
        optimizer_instance.step()
        optimizer_instance.zero_grad()

      avg_loss = total_loss / num_batches
      print(f"Loss: {avg_loss:.4f}")
